In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# Build ImageCLEF MedVQA-GI 2023 long table + label maps

This notebook standardizes the raw annotations into a single long table and builds per-question label maps.


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x


In [3]:
from pathlib import Path
import os
import sys

def find_imageclef_root() -> Path:
    env_root = os.environ.get("IMAGECLEF_MEDVQA_GI_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"IMAGECLEF_MEDVQA_GI_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "ImageCLEF_MEDVQA_GI_2023" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "ImageCLEF_MEDVQA_GI_2023" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate ImageCLEF_MEDVQA_GI_2023 root. "
        "Run from within the ImageCLEF_MEDVQA_GI_2023 folder or set IMAGECLEF_MEDVQA_GI_ROOT."
    )

ROOT = find_imageclef_root()
AUTO_CONFIG = globals().get("AUTO_CONFIG", {})
sys.path.append(str(ROOT))
OUT_DIR = ROOT / "0_dataset_prep" / "out"
LABEL_MAP_DIR = OUT_DIR / "label_maps"

# Set one of the following inputs
USE_SPLIT_FILES = bool(AUTO_CONFIG.get("use_split_files", False))  # set True if you have separate files per split


def _cfg_path(key: str, env_key: str | None = None) -> Path | None:
    val = AUTO_CONFIG.get(key) or (os.environ.get(env_key) if env_key else None)
    if not val:
        return None
    return Path(val).expanduser()


ANNOTATION_FILE = _cfg_path("annotation_file", "IMAGECLEF_MEDVQA_GI_ANN")  # single file with a 'split' column
ANNOTATION_FILES = {
    "train": _cfg_path("train_file", "IMAGECLEF_MEDVQA_GI_TRAIN"),
    "validation": _cfg_path("validation_file", "IMAGECLEF_MEDVQA_GI_VAL"),
    "test": _cfg_path("test_file", "IMAGECLEF_MEDVQA_GI_TEST"),
}

IMAGES_ROOT = _cfg_path("images_root", "IMAGECLEF_MEDVQA_GI_IMAGES")
if IMAGES_ROOT is not None:
    IMAGES_ROOT = IMAGES_ROOT.expanduser().resolve()

DEFAULT_IMAGE_EXT = ".jpg"

MAX_SAMPLES_PER_SPLIT = int(os.environ.get("MAX_SAMPLES_PER_SPLIT", "0")) or None

COLUMN_ALIASES = {
    "image_id": ["image_id", "img_id", "image", "imageid", "id", "filename"],
    "image_path": ["image_path", "path", "filepath", "image_file"],
    "question_id": ["question_id", "qid", "q_id", "questionid"],
    "question_text": ["question", "question_text", "question_str"],
    "answer_raw": ["answer", "answer_raw", "answer_text", "label"],
    "split": ["split", "subset", "set", "partition"],
}


In [4]:
from pathlib import Path
import json
import pandas as pd

from common import normalize_answer, build_label_maps, save_label_maps


def _read_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    if path.suffix.lower() in {".csv", ".tsv"}:
        return pd.read_csv(path)
    if path.suffix.lower() in {".json", ".jsonl"}:
        return pd.read_json(path, lines=path.suffix.lower() == ".jsonl")
    raise ValueError(f"Unsupported annotation format: {path}")


def _resolve_columns(df: pd.DataFrame) -> pd.DataFrame:
    col_map = {}
    for canonical, aliases in COLUMN_ALIASES.items():
        for a in aliases:
            if a in df.columns:
                col_map[a] = canonical
                break
    df = df.rename(columns=col_map)
    return df


def _ensure_columns(df: pd.DataFrame) -> pd.DataFrame:
    required = {"question_id", "question_text", "answer_raw"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    return df


# Ensure OUT_DIR / DEFAULT_ANN are available even if the config cell was skipped
def _ensure_paths():
    global OUT_DIR, DEFAULT_ANN, ANNOTATION_FILE, IMAGES_ROOT, ROOT
    if "OUT_DIR" not in globals() or "ROOT" not in globals():
        cwd = Path.cwd().resolve()
        root = None
        for p in [cwd] + list(cwd.parents):
            if p.name == "ImageCLEF_MEDVQA_GI_2023" and (p / "0_dataset_prep").exists():
                root = p
                break
        if root is None:
            for p in [cwd] + list(cwd.parents):
                if (p / "0_dataset_prep" / "out").exists():
                    root = p
                    break
        if root is None:
            raise RuntimeError("Could not locate ImageCLEF_MEDVQA_GI_2023 root. Run the config cell or set IMAGECLEF_MEDVQA_GI_ROOT.")
        ROOT = root
        OUT_DIR = root / "0_dataset_prep" / "out"

    DEFAULT_ANN = OUT_DIR / "raw" / "annotations_dev.csv"
    if "ANNOTATION_FILE" not in globals():
        ANNOTATION_FILE = None
    if "IMAGES_ROOT" not in globals():
        IMAGES_ROOT = None


def _find_dev_dataset() -> Path | None:
    _ensure_paths()
    candidates = [
        OUT_DIR / "ImageCLEFmed-MEDVQA-GI-2023-Development-Dataset",
        ROOT / "ImageCLEFmed-MEDVQA-GI-2023-Development-Dataset",
    ]
    try:
        repo_root = ROOT.parents[2]
        candidates += [
            repo_root / "Datasets" / "ImageCLEFmed-MEDVQA-GI-2023-Development-Dataset",
            repo_root / "data" / "ImageCLEFmed-MEDVQA-GI-2023-Development-Dataset",
        ]
    except Exception:
        pass

    for c in candidates:
        if c.exists():
            return c
    return None


# Auto-build annotations from ImageCLEF gt.json if needed
def _auto_build_from_gt() -> Path | None:
    _ensure_paths()
    global ANNOTATION_FILE, IMAGES_ROOT
    if DEFAULT_ANN.exists():
        ANNOTATION_FILE = DEFAULT_ANN
        return DEFAULT_ANN

    dev_root = _find_dev_dataset()
    if dev_root is None:
        return None

    gt_path = dev_root / "gt.json"
    images_root = dev_root / "images"
    if not (gt_path.exists() and images_root.exists()):
        return None

    import random

    VAL_RATIO = 0.2
    SPLIT_SEED = 42
    ANSWER_POLICY = "first"  # "first" or "join"
    QUESTION_ID_MODE = "text"  # "text" or "index"

    data = json.loads(gt_path.read_text())
    image_ids = sorted({d["ImageID"] for d in data})
    rng = random.Random(SPLIT_SEED)
    rng.shuffle(image_ids)
    n_val = int(len(image_ids) * VAL_RATIO)
    val_set = set(image_ids[:n_val])

    questions = sorted({lbl["Question"].strip() for d in data for lbl in d["Labels"]})
    qid_map = {q: i for i, q in enumerate(questions)}

    rows = []
    for d in data:
        image_id = d["ImageID"]
        split = "validation" if image_id in val_set else "train"
        for lbl in d["Labels"]:
            qtext = lbl["Question"].strip()
            ans_list = lbl.get("Answer", [])
            if not ans_list:
                ans = ""
            elif ANSWER_POLICY == "join":
                ans = " / ".join(ans_list)
            else:
                ans = ans_list[0]
            qid = qtext if QUESTION_ID_MODE == "text" else str(qid_map[qtext])
            rows.append({
                "image_id": image_id,
                "image_path": str(images_root / f"{image_id}.jpg"),
                "question_id": qid,
                "question_text": qtext,
                "answer_raw": ans,
                "split": split,
            })

    (OUT_DIR / "raw").mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(DEFAULT_ANN, index=False)
    ANNOTATION_FILE = DEFAULT_ANN
    IMAGES_ROOT = images_root
    print(f"Auto-built annotations: {DEFAULT_ANN}")
    return DEFAULT_ANN


def load_annotations() -> pd.DataFrame:
    _ensure_paths()

    if USE_SPLIT_FILES:
        missing = [k for k, v in ANNOTATION_FILES.items() if v is None or not v.exists()]
        if missing:
            raise FileNotFoundError(
                "Missing split files for: " + ", ".join(missing) +
                ". Provide AUTO_CONFIG entries or set IMAGECLEF_MEDVQA_GI_TRAIN/VAL/TEST."
            )
        frames = []
        for split, path in ANNOTATION_FILES.items():
            tmp = _read_table(path)
            tmp["split"] = split
            frames.append(tmp)
        df = pd.concat(frames, ignore_index=True)
    else:
        ann_path = ANNOTATION_FILE
        if ann_path is not None and not Path(ann_path).exists():
            ann_path = None
        if ann_path is None:
            built = _auto_build_from_gt()
            if built is None:
                raise FileNotFoundError(
                    "No annotation file found. Set AUTO_CONFIG['annotation_file'] or IMAGECLEF_MEDVQA_GI_ANN, "
                    "or place the development dataset under Datasets/ with gt.json + images."
                )
            ann_path = built
        df = _read_table(Path(ann_path))

    df = _resolve_columns(df)
    df = _ensure_columns(df)
    return df


In [5]:
# Load and standardize
raw = load_annotations()

if "split" not in raw.columns:
    raise ValueError("Expected a 'split' column or USE_SPLIT_FILES=True")

# Build image_id / image_path
if "image_path" not in raw.columns:
    if "image_id" not in raw.columns:
        raise ValueError("Expected 'image_path' or 'image_id' in annotations")
    if IMAGES_ROOT is None:
        raise FileNotFoundError(
            "image_path missing and IMAGES_ROOT not set. Set AUTO_CONFIG['images_root'] or IMAGECLEF_MEDVQA_GI_IMAGES."
        )
    if not Path(IMAGES_ROOT).exists():
        raise FileNotFoundError(f"IMAGES_ROOT not found: {IMAGES_ROOT}")
    raw["image_path"] = raw["image_id"].astype(str).apply(
        lambda x: (IMAGES_ROOT / f"{x}{DEFAULT_IMAGE_EXT}").as_posix()
    )

if "image_id" not in raw.columns:
    raw["image_id"] = raw["image_path"].astype(str).apply(lambda p: Path(p).stem)

# Normalize answers
raw["answer_raw"] = raw["answer_raw"].astype(str)
raw["answer_norm"] = raw["answer_raw"].map(normalize_answer)

# Clip to max samples per split (optional)
if MAX_SAMPLES_PER_SPLIT:
    raw = raw.groupby("split", group_keys=False).head(MAX_SAMPLES_PER_SPLIT)

# Final long table
long_df = raw[[
    "image_id",
    "image_path",
    "question_id",
    "question_text",
    "answer_raw",
    "answer_norm",
    "split",
]].copy()


Auto-built annotations: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/0_dataset_prep/out/raw/annotations_dev.csv


In [6]:
# Build label maps on train split only
label_maps = build_label_maps(long_df)

OUT_DIR.mkdir(parents=True, exist_ok=True)
LABEL_MAP_DIR.mkdir(parents=True, exist_ok=True)

# Save outputs
try:
    long_df.to_parquet(OUT_DIR / "long_table.parquet", index=False)
except Exception as e:
    print(f"Parquet save failed: {e}")
long_df.to_csv(OUT_DIR / "long_table.csv", index=False)

save_label_maps(label_maps, LABEL_MAP_DIR)

summary = {
    "n_rows": int(len(long_df)),
    "n_images": int(long_df["image_id"].nunique()),
    "n_questions": int(long_df["question_id"].nunique()),
    "splits": long_df["split"].value_counts().to_dict(),
}

with open(OUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

summary


{'n_rows': 36683,
 'n_images': 2000,
 'n_questions': 20,
 'splits': {'train': 29351, 'validation': 7332}}